### Annual and monthly climatologies of key variables 
### Results: Section 1a

In [ ]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr
import pandas as pd
import gc
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import grangercausalitytests

import warnings
warnings.filterwarnings('ignore')

In [ ]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
datac = "/Users/ellendyer/Documents/GitHub/F4R_data/"
dataf = "/Users/ellendyer/Documents/GitHub/F4R_data/analysed_fapar/"
datal = "/Users/ellendyer/Documents/GitHub/F4R_data/analysed_lai/"
datat = "/Users/ellendyer/Documents/GitHub/F4R_data/analysed_tropomi/"
dataclass = "/Volumes/New_5TB/ESA_F4R/landcover_classification/"


In [ ]:
fapar = xr.open_mfdataset(dataf+'fapar_*_reg_regrid.nc')
precip = xr.open_mfdataset(datac+'chirps_10day_reg_regrid.nc') 
lai = xr.open_mfdataset(datal+'lai_*_reg_regrid.nc') 
tropess = xr.open_mfdataset('/Users/ellendyer/Documents/GitHub/F4R_data/tropess_gridded_caf_low.nc') 
tropess = tropess.interp(lat=np.arange(tropess.coords["lat"].min().values,tropess.coords["lat"].max().values,0.25), lon=np.arange(tropess.coords["lon"].min().values,tropess.coords["lon"].max().values,0.25), method="linear")
    

In [ ]:
vclass = xr.open_mfdataset(dataclass+'*.nc') 

vclass = vclass['lccs_class'].sel(lat=slice(12,-15),lon=slice(8,31),drop=True)
vclass = vclass.astype('int').load()
vclass=vclass.where((vclass>=50) & (vclass<140),drop=True)
vclass_plot=vclass[-1,:,:]

#vclass_plot = vclass_plot.stack(flat_dim=('lat','lon'))
#
#vclass_broad_evergreen = vclass_plot.where(vclass_plot<=50,drop=True)
#vclass_broadleaved_deciduous_closed = vclass_plot.where(vclass_plot==61,drop=True)
#vclass_broadleaved_deciduous_open = vclass_plot.where(vclass_plot==62,drop=True)
#vclass_needle_evergreen_closed = vclass_plot.where(vclass_plot==71,drop=True)
#vclass_needle_evergreen_open = vclass_plot.where(vclass_plot==72,drop=True)
#vclass_needle_deciduous_closed = vclass_plot.where(vclass_plot==81,drop=True)
#vclass_needle_deciduous_open = vclass_plot.where(vclass_plot==82,drop=True)
#mosaic_tree_shrub = vclass_plot.where((vclass_plot>=100) & (vclass_plot<=101),drop=True)
#shrubland = vclass_plot.where((vclass_plot>=120) & (vclass_plot<=122),drop=True) 
#grassland = vclass_plot.where(vclass_plot==130,drop=True) 

In [ ]:
#fig, axs = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
#
#vclass_broad_evergreen.plot.scatter(x='lon',y='lat',c='green',label='broadleaf evergreen >15%',transform = ccrs.PlateCarree())
#axs.set_title('Land cover class',fontsize=10)
#axs.coastlines()
#axs.add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')
#axs.legend() 
#
#plt.suptitle("Annual Average", fontsize=10)
#plt.savefig(datap+'annual_avg.png')
#plt.show()
#plt.clf()

In [ ]:
ncols=5
nrows=1
h = nrows
w = ncols

fig, axs = plt.subplots(nrows=nrows,ncols=ncols,
                            subplot_kw={'projection': ccrs.PlateCarree()},
                            gridspec_kw = {'wspace':-0.2, 'hspace':-0.5},
                            layout="constrained",
                            figsize=(8,3))
axs=axs.flatten()

cmap=plt.cm.viridis
cmap.set_extremes(over='orange')
cmap.set_extremes(under='pink')
cst=precip['precip'].mean('time').plot.pcolormesh(ax=axs[0],
                    vmin=0, vmax=6,
                    cmap=cmap,
                    rasterized=True,
                    cbar_kwargs={"label": "","orientation":"horizontal"},
                    transform = ccrs.PlateCarree())
axs[0].set_title('Precipitation',fontsize=10)
axs[0].set_ylabel('EQ')
axs[0].coastlines()
axs[0].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

cmap=plt.cm.YlGnBu
cmap.set_extremes(over='orange')
cmap.set_extremes(under='pink')
cst=lai['LAI'].mean('time').plot.pcolormesh(ax=axs[1],
                    vmin=0, vmax=5,
                    cmap=cmap,
                    rasterized=True,
                    cbar_kwargs={"label": "","orientation":"horizontal"},
                    transform = ccrs.PlateCarree())
axs[1].set_title('LAI',fontsize=10)
axs[1].coastlines()
axs[1].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

cmap=plt.cm.ocean_r
cmap.set_extremes(over='orange')
cmap.set_extremes(under='pink')
cst=fapar['fAPAR'].mean('time').plot.pcolormesh(ax=axs[2],
                    vmin=0.25, vmax=0.8,
                    cmap=cmap,
                    rasterized=True,
                    cbar_kwargs={"label": "","orientation":"horizontal"},
                    transform = ccrs.PlateCarree())
axs[2].set_title('fAPAR',fontsize=10)
axs[2].coastlines()
axs[2].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

cmap=plt.cm.plasma_r
cmap.set_extremes(over='green')
cmap.set_extremes(under='grey')
cst=tropess['deltaD'].mean('time').plot.pcolormesh(ax=axs[3],
                    vmin=-150, vmax=-100.0,
                    cmap=cmap,
                    rasterized=True,
                    cbar_kwargs={"label": "","orientation":"horizontal"},
                    transform = ccrs.PlateCarree())
axs[3].set_title('deltaD',fontsize=10)
axs[3].coastlines()
axs[3].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

cmap=plt.cm.tab20b
cmap.set_extremes(over='orange')
cmap.set_extremes(under='pink')
cst=vclass_plot.plot.pcolormesh(ax=axs[4],
                    cmap=cmap,
                    #levels=[49.5,50.5,60.5,61.5,70,90,100,130,140],
                    rasterized=True,
                    cbar_kwargs={"label": "","orientation":"horizontal","shrink":0.8},
                    transform = ccrs.PlateCarree())
cst.colorbar.ax.set_yticklabels(cst.colorbar.ax.get_yticklabels(), rotation=45,ax=axs[4])
axs[4].add_patch(plt.Rectangle((10, 5.2), 20.7, 6.7, ls="-", lw=2, ec="purple", fc="none",zorder=3))
axs[4].add_patch(plt.Rectangle((8, -4.9), 21, 9.6, ls="-", lw=2, ec="white", fc="none",zorder=3))
axs[4].add_patch(plt.Rectangle((12, -15), 18.7, 9.8, ls="-", lw=2, ec="coral", fc="none",zorder=3))
axs[4].set_title('Land cover class',fontsize=10)
axs[4].coastlines()
axs[4].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

plt.suptitle("Annual Average", fontsize=10)
plt.savefig(datap+'annual_avg.png')
plt.show()
plt.clf()

In [ ]:
ncols=3
nrows=4
h = nrows
w = ncols
mon = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axs = plt.subplots(nrows=nrows,ncols=ncols,
                            subplot_kw={'projection': ccrs.PlateCarree()},
                            gridspec_kw = {'wspace':-0.2, 'hspace':-0.5},
                            layout="constrained",
                            figsize=(4,5))
axs=axs.flatten()
var = fapar['fAPAR'].groupby('time.month').mean('time')

cmap=plt.cm.ocean_r
cmap.set_extremes(over='orange')
cmap.set_extremes(under='pink')

for i in range(0,12):
    cst=var[i,:,:].plot.pcolormesh(ax=axs[i],
                        vmin=0.25, vmax=0.8,
                        cmap=cmap,
                        rasterized=True,
                        add_colorbar=False,
                        #cbar_kwargs={"label": ""},
                        transform = ccrs.PlateCarree())
    if i==0:
        axs[0].add_patch(plt.Rectangle((10, 5.2), 20.7, 6.7, ls="-", lw=1, ec="purple", fc="none",zorder=3))
        axs[0].add_patch(plt.Rectangle((8, -4.9), 21, 9.6, ls="-", lw=1, ec="white", fc="none",zorder=3))
        axs[0].add_patch(plt.Rectangle((12, -15), 18.7, 9.8, ls="-", lw=1, ec="coral", fc="none",zorder=3))
    axs[i].set_title(mon[i],fontsize=10)
    axs[i].coastlines()
    axs[i].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

fig.subplots_adjust(hspace=0,wspace=-0.5)
cbar = fig.colorbar(cst, ax=axs, orientation='horizontal',
                        #location="bottom",
                        shrink=0.3,extend="both",fraction=0.2)

plt.suptitle("fAPAR", fontsize=10)
plt.savefig(datap+'mon_clim_fapar.png')
plt.show()
plt.clf()

In [ ]:
ncols=3
nrows=4
h = nrows
w = ncols
mon = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axs = plt.subplots(nrows=nrows,ncols=ncols,
                            subplot_kw={'projection': ccrs.PlateCarree()},
                            gridspec_kw = {'wspace':-0.2, 'hspace':-0.5},
                            layout="constrained",
                            figsize=(4,5))
axs=axs.flatten()
var = lai['LAI'].groupby('time.month').mean('time')

cmap=plt.cm.YlGnBu
cmap.set_extremes(over='orange')
cmap.set_extremes(under='pink')

for i in range(0,12):
    cst=var[i,:,:].plot.pcolormesh(ax=axs[i],
                        vmin=0.0, vmax=5.0,
                        cmap=cmap,
                        rasterized=True,
                        add_colorbar=False,
                        #cbar_kwargs={"label": ""},
                        transform = ccrs.PlateCarree())
    if i==0:
        axs[0].add_patch(plt.Rectangle((10, 5.2), 20.7, 6.7, ls="-", lw=1, ec="purple", fc="none",zorder=3))
        axs[0].add_patch(plt.Rectangle((8, -4.9), 21, 9.6, ls="-", lw=1, ec="white", fc="none",zorder=3))
        axs[0].add_patch(plt.Rectangle((12, -15), 18.7, 9.8, ls="-", lw=1, ec="coral", fc="none",zorder=3))
    axs[i].set_title(mon[i],fontsize=10)
    axs[i].coastlines()
    axs[i].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

fig.subplots_adjust(hspace=0,wspace=-0.5)
cbar = fig.colorbar(cst, ax=axs, orientation='horizontal',
                        #location="bottom",
                        shrink=0.3,extend="both",fraction=0.2)

plt.suptitle("LAI", fontsize=10)
plt.savefig(datap+'mon_clim_lai.png')
plt.show()
plt.clf()

In [ ]:
ncols=3
nrows=4
h = nrows
w = ncols
mon = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axs = plt.subplots(nrows=nrows,ncols=ncols,
                            subplot_kw={'projection': ccrs.PlateCarree()},
                            gridspec_kw = {'wspace':-0.2, 'hspace':-0.5},
                            layout="constrained",
                            figsize=(4,5))
axs=axs.flatten()
var = precip['precip'].groupby('time.month').mean('time')

cmap=plt.cm.viridis
cmap.set_extremes(over='orange')
cmap.set_extremes(under='pink')

for i in range(0,12):
    cst=var[i,:,:].plot.pcolormesh(ax=axs[i],
                        vmin=0.0, vmax=8.0,
                        cmap=cmap,
                        rasterized=True,
                        add_colorbar=False,
                        #cbar_kwargs={"label": ""},
                        transform = ccrs.PlateCarree())
    if i==0:
        axs[0].add_patch(plt.Rectangle((10, 5.2), 20.7, 6.7, ls="-", lw=1, ec="purple", fc="none",zorder=3))
        axs[0].add_patch(plt.Rectangle((8, -4.9), 21, 9.6, ls="-", lw=1, ec="white", fc="none",zorder=3))
        axs[0].add_patch(plt.Rectangle((12, -15), 18.7, 9.8, ls="-", lw=1, ec="coral", fc="none",zorder=3))
    axs[i].set_title(mon[i],fontsize=10)
    axs[i].coastlines()
    axs[i].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

fig.subplots_adjust(hspace=0,wspace=-0.5)
cbar = fig.colorbar(cst, ax=axs, orientation='horizontal',
                        #location="bottom",
                        shrink=0.3,extend="both",fraction=0.2)

plt.suptitle("Precip", fontsize=10)
plt.savefig(datap+'mon_clim_precip.png')
plt.show()
plt.clf()

In [ ]:
ncols=3
nrows=4
h = nrows
w = ncols
mon = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axs = plt.subplots(nrows=nrows,ncols=ncols,
                            subplot_kw={'projection': ccrs.PlateCarree()},
                            gridspec_kw = {'wspace':-0.2, 'hspace':-0.5},
                            layout="constrained",
                            figsize=(4,5))
axs=axs.flatten()
var = tropess['deltaD'].groupby('time.month').mean('time')

cmap=plt.cm.plasma_r
cmap.set_extremes(over='green')
cmap.set_extremes(under='grey')

for i in range(0,12):
    cst=var[i,:,:].plot.pcolormesh(ax=axs[i],
                        vmin=-150, vmax=-100.0,
                        cmap=cmap,
                        rasterized=True,
                        add_colorbar=False,
                        #cbar_kwargs={"label": ""},
                        transform = ccrs.PlateCarree())
    if i==0:
        axs[0].add_patch(plt.Rectangle((10, 5.2), 20.7, 6.7, ls="-", lw=1, ec="purple", fc="none",zorder=3))
        axs[0].add_patch(plt.Rectangle((8, -4.9), 21, 9.6, ls="-", lw=1, ec="white", fc="none",zorder=3))
        axs[0].add_patch(plt.Rectangle((12, -15), 18.7, 9.8, ls="-", lw=1, ec="coral", fc="none",zorder=3))
    axs[i].set_title(mon[i],fontsize=10)
    axs[i].coastlines()
    axs[i].add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')

fig.subplots_adjust(hspace=0,wspace=-0.5)
cbar = fig.colorbar(cst, ax=axs, orientation='horizontal',
                        #location="bottom",
                        shrink=0.3,extend="both",fraction=0.2)

plt.suptitle("deltaD", fontsize=10)
plt.savefig(datap+'mon_clim_deltaD.png')
plt.show()
plt.clf()